In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tensorflow.keras.models import load_model # Fixed: load model -> load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load trained model
model = load_model('vgg16_face_recognition_model.h5')

# imageDatagenerator automatically loads images and performs preprocessing
datagen = ImageDataGenerator(

    #Normalize pixel values between 0 and 1
    rescale=1./255,

    #Rescale 20% of dataset for validation
    validation_split=0.2 # Fixed: 0,2 -> 0.2
    )

train_data = datagen.flow_from_directory(
    'dataset',
    target_size=(224, 224),     # Resize images to vgg16 input size
    batch_size=32,             # number of images processed per iteration
    class_mode='categorical',  # Fixed: Added 'categorical' for multiclass classification
    subset='training'  #Load training split

)

# get class labels from dataset
labels = list(train_data.class_indices.keys())


#Load face detection model
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

#start webcam

cap=cv2.VideoCapture(0)

while True:

  # capture frame
  ret,frame=cap.read()

  # convert to grayscale
  gray =  cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)

  # detect faces
  faces = face_cascade.detectMultiScale(gray,1.3,5)

  for (x,y,w,h) in faces:

    # crop detected face
    face =  frame[y:y+h,  x:x+w]


    #Resize for VGG16
    face = cv2.resize(face,(224,224))

    #Normalize pixel values
    face = face/255.0

    # convert to 40 tensor
    face = np.reshape(face,(1,224,224,3))

    #predict person
    pred  = model.predict(face)

    #get label with highest probability
    label = labels[np.argmax(pred)]

    #Display predicted name
    cv2.putText(frame,label,(x,y-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,(255,0,0),2)

    # Draw face rectangle
    cv2.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),2)

  #show frame
  cv2.imshow('Face Recognition',frame)

  # Exit when ESC pressed
  if cv2.waitKey(1)==27: # Fixed: waitkey -> waitKey and corrected indentation
    break

# release webcam
cap.release()

# close windows
cv2.destroyAllWindows()

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'face_recognition_vgg16.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)